# Neural Receiver for Weak Signal Detection and Parametric Classification
## Google Colab Setup

이 노트북은 Google Colab에서 프로젝트를 실행하기 위한 환경 설정을 제공합니다.

### 1. GPU 설정 확인
런타임 > 런타임 유형 변경 > 하드웨어 가속기 > GPU 선택

In [ ]:
# GPU 사용 가능 여부 확인
import torch
print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 이름: {torch.cuda.get_device_name(0)}")
    print(f"GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

### 2. GitHub 저장소 클론

In [ ]:
# 저장소 클론 (이미 클론되어 있다면 스킵)
import os

repo_name = "Neural-Receiver-for-Weak-Signal-Detection-and-Parametric-Classification"
repo_url = "https://github.com/hyeonhwilee/Neural-Receiver-for-Weak-Signal-Detection-and-Parametric-Classification.git"

if not os.path.exists(repo_name):
    !git clone {repo_url}
    print(f"✓ {repo_name} 클론 완료")
else:
    print(f"✓ {repo_name} 이미 존재")

# 프로젝트 디렉토리로 이동
%cd {repo_name}

### 3. 필요한 패키지 설치

In [ ]:
# requirements.txt가 있는 경우
if os.path.exists('requirements.txt'):
    !pip install -q -r requirements.txt
    print("✓ requirements.txt 패키지 설치 완료")
else:
    # 기본 패키지 설치
    !pip install -q torch torchvision torchaudio
    !pip install -q numpy scipy matplotlib seaborn
    !pip install -q scikit-learn pandas
    !pip install -q tqdm tensorboard
    print("✓ 기본 패키지 설치 완료")

### 4. 라이브러리 임포트 및 설정

In [ ]:
# 기본 라이브러리
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

# 시각화 설정
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# 랜덤 시드 설정
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    
set_seed(42)
print("✓ 환경 설정 완료")

### 5. Google Drive 마운트 (선택사항)
데이터셋이나 모델을 저장하려면 Drive를 마운트하세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive 마운트 완료")

### 6. IQ 신호 생성 예제
약한 신호 검출을 위한 기본 IQ(In-phase/Quadrature) 신호 생성

In [ ]:
def generate_iq_signal(num_samples=1000, frequency=5, noise_level=0.1, snr_db=-10):
    """
    IQ 신호 생성 함수
    
    Args:
        num_samples: 샘플 수
        frequency: 신호 주파수
        noise_level: 노이즈 레벨
        snr_db: Signal-to-Noise Ratio (dB)
    
    Returns:
        iq_signal: 복소수 IQ 신호 (I + jQ)
    """
    t = np.linspace(0, 1, num_samples)
    
    # 신호 생성
    signal = np.exp(1j * 2 * np.pi * frequency * t)
    
    # SNR에 따른 신호 파워 조정
    snr_linear = 10 ** (snr_db / 10)
    signal_power = np.sqrt(snr_linear)
    signal = signal * signal_power
    
    # 노이즈 추가 (가우시안 복소 노이즈)
    noise_i = np.random.normal(0, noise_level, num_samples)
    noise_q = np.random.normal(0, noise_level, num_samples)
    noise = noise_i + 1j * noise_q
    
    iq_signal = signal + noise
    
    return iq_signal

# 예제 신호 생성
iq_signal = generate_iq_signal(num_samples=1000, snr_db=-10)

# IQ 신호 시각화
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# I 성분
axes[0, 0].plot(iq_signal.real[:200])
axes[0, 0].set_title('I (In-phase) Component')
axes[0, 0].set_xlabel('Sample')
axes[0, 0].set_ylabel('Amplitude')
axes[0, 0].grid(True)

# Q 성분
axes[0, 1].plot(iq_signal.imag[:200])
axes[0, 1].set_title('Q (Quadrature) Component')
axes[0, 1].set_xlabel('Sample')
axes[0, 1].set_ylabel('Amplitude')
axes[0, 1].grid(True)

# IQ 평면 (Constellation)
axes[1, 0].scatter(iq_signal.real, iq_signal.imag, alpha=0.3, s=1)
axes[1, 0].set_title('IQ Constellation')
axes[1, 0].set_xlabel('I')
axes[1, 0].set_ylabel('Q')
axes[1, 0].grid(True)
axes[1, 0].axis('equal')

# 스펙트럼
fft = np.fft.fft(iq_signal)
freq = np.fft.fftfreq(len(iq_signal))
axes[1, 1].plot(freq, 20 * np.log10(np.abs(fft)))
axes[1, 1].set_title('Frequency Spectrum')
axes[1, 1].set_xlabel('Normalized Frequency')
axes[1, 1].set_ylabel('Magnitude (dB)')
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

print(f"✓ IQ 신호 생성 완료: {len(iq_signal)} samples")

### 7. 간단한 신경망 수신기 예제

In [ ]:
class NeuralReceiver(nn.Module):
    """
    약한 신호 검출 및 분류를 위한 간단한 신경망 수신기
    """
    def __init__(self, input_size=2, hidden_size=64, num_classes=2):
        super(NeuralReceiver, self).__init__()
        
        self.feature_extractor = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_size),
            nn.Dropout(0.3),
            
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_size),
            nn.Dropout(0.3),
        )
        
        # 신호 검출 헤드
        self.detector = nn.Linear(hidden_size, 1)
        
        # 파라미터 분류 헤드
        self.classifier = nn.Linear(hidden_size, num_classes)
        
    def forward(self, x):
        # x shape: (batch_size, 2) - I와 Q 성분
        features = self.feature_extractor(x)
        
        # 신호 검출 (이진 분류: 신호 있음/없음)
        detection = torch.sigmoid(self.detector(features))
        
        # 파라미터 분류
        classification = self.classifier(features)
        
        return detection, classification

# 모델 생성 및 정보 출력
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = NeuralReceiver(input_size=2, hidden_size=64, num_classes=4).to(device)

print("✓ Neural Receiver 모델 생성 완료")
print(f"  - Device: {device}")
print(f"  - Parameters: {sum(p.numel() for p in model.parameters()):,}")
print("\n모델 구조:")
print(model)

### 8. 프로젝트 파일 목록 확인

In [ ]:
# 프로젝트 파일 구조 확인
!ls -la

print("\n" + "="*50)
print("환경 설정이 완료되었습니다!")
print("이제 프로젝트 코드를 실행할 수 있습니다.")
print("="*50)